In [ ]:
# pyright: reportGeneralTypeIssues=false, reportUnknownMemberType=false, reportUnknownVariableType=false, reportUnknownArgumentType=false
# ruff: noqa
# pylint: skip-file

# NHANES Diabetes Prediction - Bayesian Hyperparameter Optimization

This notebook implements Bayesian optimization using Optuna with W&B tracking for the LightGBM diabetes screening model.

**Optimization target**: Recall (prioritizing detection of positive cases for screening purposes)

## 1. Setup and Configuration

In [ ]:
import lightgbm as lgb
import matplotlib.pyplot as plt

import optuna
import pandas as pd
import wandb
from optuna.integration.wandb import WeightsAndBiasesCallback
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    recall_score,
)
from sklearn.model_selection import train_test_split  # pyright: ignore[reportUnknownVariableType]
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import precision_recall_curve, auc, recall_score, precision_score

import numpy as np

In [ ]:
# Configuration

# Weights and Biases
WANDB_PROJECT = "Model exploration for Diabetes Prediction"
ENTITY = "fastegiano-tesis"


# Optuna
STORAGE = "sqlite:///optuna_diabetes.db"
STUDY_NAME = "lgbm-CVrecall + ES/Bagging/FeatFrac/is_unbalance"
PROJECT_NAME = "Model exploration for Diabetes Prediction"
RANDOM_STATE = 37
N_TRIALS = 100

# Categorical features
CAT_FEATURES = [
    "education_level",
    "moderate_every_X_days",
    "vigorous_every_X_days",
    "has_partner",
    "had_partner",
    "is_female",
    "ever_smoker",
    "is_current_smoker",
    "drinking_frequency",
]

## 2. Data Loading and Preparation

In [ ]:
data = pd.read_csv("../../../dataset/processed_data.csv")  # pyright: ignore[reportUnknownMemberType]
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
# Drop WTMEC2YR if present (no error if absent) and show new shape
data.drop(columns="WTMEC2YR", inplace=True, errors="ignore")
print(f"Data shape: {data.shape}")

In [ ]:
X = data.iloc[:, :-1]
y = data.iloc[:, -1]

X_train, X_test, y_train, y_test = train_test_split(  # pyright: ignore[reportUnknownVariableType]
    X, y, test_size=0.1, stratify=y, random_state=RANDOM_STATE
)

In [ ]:
y_test.value_counts()

In [ ]:
y_train.value_counts()

### Null revision

In [ ]:
# Check for null values in training and test sets
print("=" * 70)
print("NULL VALUES SUMMARY")
print("=" * 70)

# Create a summary DataFrame
null_summary = pd.DataFrame({
    "X_train_null_%": (X_train.isnull().sum() / len(X_train)) * 100,
    "X_test_null_%": (X_test.isnull().sum() / len(X_test)) * 100,
})

# Filter to only show columns with any nulls (optional)
null_summary = null_summary[
    (null_summary["X_train_null_%"] > 0) | (null_summary["X_test_null_%"] > 0)
]

# Sort by X_train nulls descending
null_summary = null_summary.sort_values("X_train_null_%", ascending=False)

print(f"\nX_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(
    f"\nTotal null values - X_train: {X_train.isnull().sum().sum()}, X_test: {X_test.isnull().sum().sum()}"
)

if len(null_summary) > 0:
    print(f"\nColumns with null values:\n")
    print(null_summary.to_string())
else:
    print("\nNo null values found in either dataset!")

print("\n" + "=" * 70)

---

## 3. Model preparation

## Weights & Biases initialization

In [ ]:
WANDB_KWARGS = {  # type: ignore
    "entity": ENTITY,
    "project": WANDB_PROJECT,
    "name": STUDY_NAME,
    "config": {
        "random_state": RANDOM_STATE,
        "n_trials": N_TRIALS,
        "metric": "recall",
    },
}

## Optimization set up

In [ ]:
def objective(trial):  # type: ignore

    # Suggest max_depth first since num_leaves depends on it
    max_depth = trial.suggest_int("max_depth", 3, 8)  # type: ignore
    num_leaves = int(2**max_depth * 1.15)  # type: ignore

    params = {  # pyright: ignore[reportUnknownVariableType]
        "objective": "binary",
        "metric": "binary_logloss",
        "verbosity": -1,
        "random_state": RANDOM_STATE,
        "n_estimators": 2000,  # trial.suggest_int("n_estimators", 100, 1000), # type: ignore
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),  # type: ignore
        "max_depth": max_depth,  # type: ignore
        "num_leaves": num_leaves,  # type: ignore
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 100),  # type: ignore
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.1, 1.0),  # type: ignore
        "feature_fraction": trial.suggest_float("feature_fraction", 0.1, 1.0),  # type: ignore
        "is_unbalance": True,
        "bagging_seed": RANDOM_STATE,
        "feature_fraction_seed": RANDOM_STATE,
    }

    kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
    val_scores = []
    train_scores = []

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):  # type: ignore
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]  # type: ignore
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]  # type: ignore

        model = lgb.LGBMClassifier(**params)  # type: ignore
        model.fit(  # type: ignore
            X_tr,
            y_tr,  # type: ignore
            eval_set=[(X_val, y_val)],
            callbacks=[lgb.early_stopping(50, verbose=False)],
            categorical_feature=CAT_FEATURES,
        )

        # Calculate validation metrics
        y_val_pred = model.predict(X_val)  # type: ignore
        val_recall = recall_score(y_val, y_val_pred)  # type: ignore
        val_scores.append(val_recall)

        # Calculate training metrics (on same fold data)
        y_train_pred = model.predict(X_tr)  # type: ignore
        train_recall = recall_score(y_tr, y_train_pred)  # type: ignore
        train_scores.append(train_recall)

    # Calculate mean metrics across folds
    mean_val_recall = np.mean(val_scores)  # type: ignore
    mean_train_recall = np.mean(train_scores)  # type: ignore
    overfitting_gap = mean_train_recall - mean_val_recall

    # Log aggregated trial metrics to W&B with explicit step
    wandb.log(
        {  # type: ignore
            "mean_train_recall": mean_train_recall,
            "mean_val_recall": mean_val_recall,
            "overfitting_gap": overfitting_gap,
            "std_val_recall": np.std(val_scores),  # type: ignore
            "std_train_recall": np.std(train_scores),  # type: ignore
        },
        step=trial.number,
    )

    return mean_val_recall

## 4. Run Optimization

In [ ]:
# Delete all studies
"""
optuna.delete_study(
    study_name=STUDY_NAME,
    storage="sqlite:///optuna.db"  # change if your storage is different
)
"""

In [ ]:
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    load_if_exists=True,
    direction="maximize",
)

In [ ]:
wandb_callback = WeightsAndBiasesCallback(
    metric_name="recall",
    wandb_kwargs=WANDB_KWARGS,  # type: ignore
)

study.optimize(objective, n_trials=N_TRIALS, callbacks=[wandb_callback])  # type: ignore

wandb.finish()

## 5. Results Analysis

In [ ]:
# Display best params from the Optuna study
best_params = study.best_params
print("Best parameters:")
for k, v in best_params.items():
    print(f"{k}: {v}")

print(f"\nBest recall (study.best_value): {study.best_value:.4f}")

## 6. Best Model Evaluation

In [ ]:
# 1️⃣ Split TRAIN into train / valid (internal, not test)
X_tr, X_val, y_tr, y_val = train_test_split(  # type: ignore
    X_train,
    y_train,  # type: ignore
    test_size=0.1,
    stratify=y_train,  # type: ignore
    random_state=RANDOM_STATE,
)

best_model = lgb.LGBMClassifier(**study.best_params, random_state=RANDOM_STATE)

best_model.fit(  # type: ignore
    X_tr,
    y_tr,  # type: ignore
    eval_set=[(X_tr, y_tr), (X_val, y_val)],
    eval_names=["train", "valid"],
    eval_metric="binary_logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=20)],
    categorical_feature=CAT_FEATURES,
)  # type: ignore

y_pred = best_model.predict(X_test)  # type: ignore

In [ ]:
recall_test_positive = np.round(recall_score(y_test, y_pred, pos_label=1), 2)  # type: ignore
recall_test_negative = np.round(recall_score(y_test, y_pred, pos_label=0), 2)  # type: ignore

In [ ]:
print(f"y_test value counts:\n{y_test.value_counts()}")
print(f"y_pred value counts:\n{pd.Series(y_pred).value_counts()}")
print(f"Recall test (pos_label=1): {recall_test_positive}")
print(f"Recall test (pos_label=0): {recall_test_negative}")

### Training Progress - Binary Log Loss

In [ ]:
# Extract evaluation results
results = best_model.evals_result_

# Plot the progression
plt.figure(figsize=(10, 6))  # type: ignore
plt.plot(results["train"]["binary_logloss"], label="Train", linewidth=2)  # type: ignore
plt.plot(results["valid"]["binary_logloss"], label="Valid", linewidth=2)  # type: ignore
plt.xlabel("Iteration")  # type: ignore
plt.ylabel("Binary Log Loss")  # type: ignore
plt.title("Binary Log Loss Over Training Iterations")  # type: ignore
plt.legend()  # type: ignore
plt.grid(alpha=0.3)  # type: ignore
plt.tight_layout()
plt.show()  # type: ignore

### Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)  # type: ignore
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["No Diabetes", "Diabetes"]
)
disp.plot(cmap="Blues")  # type: ignore
plt.title(f"Recall: {recall_test_positive}")  # type: ignore
plt.tight_layout()
plt.show()  # type: ignore

## 8. Feature Importance

In [ ]:
fi = pd.Series(best_model.feature_importances_, index=X.columns)
fi_sorted = fi.sort_values(ascending=True)

plt.figure(figsize=(10, 8))  # type: ignore
fi_sorted.plot(kind="barh")
plt.title("Feature Importances Splits (LightGBM)")  # type: ignore
plt.xlabel("Importance")  # type: ignore
plt.tight_layout()
plt.show()  # type: ignore

In [ ]:
fi = pd.Series(
    best_model.booster_.feature_importance(importance_type="gain"), index=X.columns
)
fi_sorted = fi.sort_values(ascending=True)

plt.figure(figsize=(10, 8))  # type: ignore
fi_sorted.plot(kind="barh")
plt.title("Feature Importances GAIN (LightGBM)")  # type: ignore
plt.xlabel("Importance")  # type: ignore
plt.tight_layout()
plt.show()  # type: ignore

# Post Hoc - Threshold optimization

In [ ]:
# Assuming you have y_test and model already
y_proba = best_model.predict_proba(X_test)[:, 1]

# Calculate PR curve
precisions, recalls, thresholds = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recalls, precisions)

# Create figure with 2 subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Precision-Recall Curve ---
ax1 = axes[0]
ax1.plot(recalls, precisions, "b-", linewidth=2, label=f"PR Curve (AUC={pr_auc:.3f})")
ax1.fill_between(recalls, precisions, alpha=0.2)

# Mark key threshold points
target_recalls = [0.80, 0.75, 0.70, 0.50]
colors = ["red", "orange", "green", "purple"]

for target, color in zip(target_recalls, colors):
    idx = np.where(recalls[:-1] >= target)[0]
    if len(idx) > 0:
        i = idx[-1]
        thresh = thresholds[i]
        ax1.scatter(
            recalls[i],
            precisions[i],
            c=color,
            s=100,
            zorder=5,
            label=f"Recall={target:.0%} (thresh={thresh:.3f}, prec={precisions[i]:.1%})",
        )

# Baseline (random classifier)
baseline = y_test.mean()
ax1.axhline(
    y=baseline,
    color="gray",
    linestyle="--",
    label=f"Baseline (prevalence={baseline:.1%})",
)

ax1.set_xlabel("Recall (Sensitivity)", fontsize=12)
ax1.set_ylabel("Precision (PPV)", fontsize=12)
ax1.set_title("Precision-Recall Curve", fontsize=14)
ax1.legend(loc="upper right", fontsize=9)
ax1.set_xlim([0, 1.02])
ax1.set_ylim([0, 1.02])
ax1.grid(True, alpha=0.3)

# --- Plot 2: Threshold vs Metrics ---
ax2 = axes[1]

# Calculate recall and precision at each threshold
thresh_range = np.linspace(0.05, 0.6, 100)
recall_at_thresh = []
precision_at_thresh = []
f1_at_thresh = []
flagged_pct = []

for t in thresh_range:
    y_pred = (y_proba >= t).astype(int)
    r = recall_score(y_test, y_pred, zero_division=0)
    p = precision_score(y_test, y_pred, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    recall_at_thresh.append(r)
    precision_at_thresh.append(p)
    f1_at_thresh.append(f1)
    flagged_pct.append(y_pred.mean())

ax2.plot(thresh_range, recall_at_thresh, "b-", linewidth=2, label="Recall")
ax2.plot(thresh_range, precision_at_thresh, "r-", linewidth=2, label="Precision")
ax2.plot(thresh_range, f1_at_thresh, "g--", linewidth=2, label="F1 Score")
ax2.plot(thresh_range, flagged_pct, "k:", linewidth=2, label="% Flagged")

# Mark default 0.5 threshold
ax2.axvline(x=0.5, color="gray", linestyle="--", alpha=0.7, label="Default (0.5)")

# Mark optimal threshold for 80% recall
target_80_idx = np.argmin(np.abs(np.array(recall_at_thresh) - 0.80))
optimal_thresh = thresh_range[target_80_idx]
ax2.axvline(
    x=optimal_thresh,
    color="red",
    linestyle="--",
    alpha=0.7,
    label=f"80% Recall (thresh={optimal_thresh:.3f})",
)

ax2.set_xlabel("Decision Threshold", fontsize=12)
ax2.set_ylabel("Score", fontsize=12)
ax2.set_title("Metrics vs Decision Threshold", fontsize=14)
ax2.legend(loc="center right", fontsize=9)
ax2.set_xlim([0.05, 0.6])
ax2.set_ylim([0, 1.02])
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("precision_recall_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Print Summary Table ---
print("\n" + "=" * 70)
print("THRESHOLD ANALYSIS SUMMARY")
print("=" * 70)
print(f"{'Threshold':<12} {'Recall':<12} {'Precision':<12} {'F1':<12} {'Flagged':<12}")
print("-" * 70)

for thresh in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    y_pred = (y_proba >= thresh).astype(int)
    r = recall_score(y_test, y_pred, zero_division=0)
    p = precision_score(y_test, y_pred, zero_division=0)
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    n_flagged = y_pred.sum()
    pct_flagged = y_pred.mean() * 100
    print(
        f"{thresh:<12.2f} {r:<12.1%} {p:<12.1%} {f1:<12.3f} {n_flagged} ({pct_flagged:.1f}%)"
    )

print("=" * 70)
print(
    f"\nTest set: {len(y_test)} samples, {y_test.sum()} diabetes cases ({y_test.mean():.1%} prevalence)"
)

In [ ]:
t = 0.18

y_pred_final = (y_proba >= t).astype(int)

In [ ]:
r = recall_score(y_test, y_pred_final, zero_division=0)  # type: ignore
p = precision_score(y_test, y_pred_final, zero_division=0)  # type: ignore
f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

In [ ]:
print(
    f"\nFinal chosen threshold: {t}"
    f"\n - Recall: {r:.2%}"
    f"\n - Precision: {p:.2%}"
    f"\n - F1 Score: {f1:.3f}"
)

In [ ]:
cm = confusion_matrix(y_test, y_pred_final)  # type: ignore
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["No Diabetes", "Diabetes"]
)
disp.plot(cmap="Blues")  # type: ignore
plt.title(f"Recall: {r:.2%}")  # type: ignore
plt.tight_layout()
plt.show()  # type: ignore